# CS 3892 / 5892 — Session 24 · Neural-network verification I

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-11-19-nn-verification-robustness.ipynb)

**Thursday, November 19, 2026.** Local robustness as a formal query, and three ways to answer it —
**IBP**, **CROWN**, and the **exact** answer — then the **ε frontier** and **branch and bound**, on a
network small enough to check by hand: 2 inputs → 2 ReLU neurons → 2 outputs (`python/nnv.py`).
The hand-written IBP and CROWN are checked against **auto_LiRPA**, HW5's tool. HW5 goes out after
class; this session covers its Parts 1, 2 and 4. (HW5's own network is different.)

## Setup

Run these two once. The second installs HW5's pinned tools: a minute or two.

In [ ]:
# --- Setup: find the repo (clone on Colab), define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-11-19-nn-verification-robustness"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

PYD = ROOT / "sessions" / SESSION / "python"
HD  = ROOT / "homework" / "hw5-nn-verification"
FIG = ROOT / "sessions" / SESSION / "figures"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- The tools, HW5's pinned set: torch 2.11.0 CPU (about 200 MB), auto_LiRPA 0.8.0 at its release
# commit, numpy. A minute or two the first time.
pip = [sys.executable, "-m", "pip", "install", "--quiet"]
subprocess.run(pip + ["-r", str(HD / "requirements.txt")], check=True)
subprocess.run(pip + ["--no-deps", "--ignore-requires-python", "-r", str(HD / "requirements-lirpa.txt")], check=True)
import torch, auto_LiRPA
print("torch", torch.__version__, "| auto_LiRPA", auto_LiRPA.__version__)

## 1. Robustness as a formal query

At x0 = (0.45, 0.55) class 0 wins with margin 0.4. Robust at radius ε means: for EVERY x within ε,
class 0 still wins. VNN-LIB writes the counterexample region; robust iff it is empty.

In [ ]:
show(PYD / "nnv.py")
out = run(PYD / "01_robustness_query.py")
assert "(assert (>= Y_1 Y_0))" in out

## 2. IBP, CROWN, and the exact answer

Both IBP and CROWN are sound (a positive bound proves robustness) and incomplete (a negative bound
means "unknown"). The exact minimum decides.

In [ ]:
out = run(PYD / "02_ibp_crown_exact.py")
assert "unknown / verified / verified" in out and "falsified" in out

## 3. The ε frontier, and branch and bound

Each method's largest provable radius; splitting the box until CROWN verifies every piece.

In [ ]:
out = run(PYD / "03_frontier_and_bab.py")
assert "IBP 0.040   CROWN 0.125   true robust radius 0.200" in out

## Try it yourself

Edit `python/nnv.py` (folder icon, left), save, and re-run a cell.

- Move x0 to (0.6, 0.5). Which method's frontier changes most, and why? (Look at which neurons are
  unstable.)
- In `relax`, always use lower slope 0. What happens to CROWN's frontier? That choice is what
  α-CROWN optimises.
- Give `bab` a smaller `depth`. At ε = 0.2, the true minimum margin is exactly 0: what does branch
  and bound report, and why can it never say "verified" there?